# Rathipriya et al. (2023) — Daily PharmaSales Final-Pipeline Alignment

Notebook ini eksplorasi preprocessing berdasarkan penelitian sebelumnya. Fokusnya menjalankan baseline Rathi Priya daily PharmaSales dengan preprocessing pipeline dari penelitian kami, bukan dari paper referensi (paper referensi belum dikirim).

**Pipeline:**
- Dataset: `data/raw/pharma-sales/salesdaily.csv`
- Granularitas: daily
- Unit observasi: 8 kategori ATC
- Lag selection: ACF per kategori, `max_lag=26`
- Feature: `lag_1 ... lag_n` + `rolling_mean_n` dengan `shift(1)`
- Split: chronological `80% train / 20% test`
- Final fit: train only
- Target scale: original daily sales scale, tanpa target scaling
- Metric: MSE dan RMSE


In [14]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from statsmodels.tsa.stattools import acf

try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_AVAILABLE = True
except Exception:
    ARIMA = None
    STATSMODELS_AVAILABLE = False

CATEGORIES = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
MAX_LAG = 26
RANDOM_STATE = 42

BANDWIDTHS = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
RBF_CENTERS = [5, 10, 20, 40]
RBF_GAMMAS = [0.01, 0.05, 0.1, 0.5, 1.0]
RBF_ALPHAS = [0.01, 0.1, 1.0]

# Reference-Based Exploration

Eksplorasi mengikuti metodologi dari paper Rathipriya et al. (2023). Preprocessing: split chronologis 70/15/15 (train/validation/test), feature basic lag_1 (tanpa ACF selection dan rolling mean). Hyperparameter tuning dilakukan pada validation set, evaluasi final pada test set.


## Preprocessing

Preprocessing mengikuti paper: data di-load, kemudian di-split secara chronologis 70% train / 15% validation / 15% test. Feature yang digunakan hanya `lag_1` (y(t-1)) per kategori — tanpa ACF-based lag selection, tanpa rolling mean, berbeda dengan preprocessing kita di section selanjutnya.


In [15]:
ref_categories = []

for category in CATEGORIES:
    dfg = df[["datum", category]].rename(columns={"datum": "ds", category: "y"}).copy()
    dfg["lag_1"] = dfg["y"].shift(1)
    dfg = dfg.dropna().reset_index(drop=True)

    train_size = int(len(dfg) * 0.70)
    val_size = int(len(dfg) * 0.15)

    train_df = dfg.iloc[:train_size]
    val_df = dfg.iloc[train_size:train_size + val_size]
    test_df = dfg.iloc[train_size + val_size:]

    feature_columns = ["lag_1"]

    ref_categories.append({
        "Category": category,
        "feature_columns": feature_columns,
        "X_train": train_df[feature_columns].to_numpy(dtype=float),
        "y_train": train_df["y"].to_numpy(dtype=float),
        "X_val": val_df[feature_columns].to_numpy(dtype=float),
        "y_val": val_df["y"].to_numpy(dtype=float),
        "X_test": test_df[feature_columns].to_numpy(dtype=float),
        "y_test": test_df["y"].to_numpy(dtype=float),
    })

pd.DataFrame([
    {
        "Category": item["Category"],
        "train_size": len(item["X_train"]),
        "val_size": len(item["X_val"]),
        "test_size": len(item["X_test"]),
    }
    for item in ref_categories
])


,Category,train_size,val_size,test_size
0,M01AB,1473,315,317
1,M01AE,1473,315,317
2,N02BA,1473,315,317
3,N02BE,1473,315,317
4,N05B,1473,315,317
5,N05C,1473,315,317
6,R03,1473,315,317
7,R06,1473,315,317


### GRNN

General Regression Neural Network — kernel regression dengan Gaussian kernel. Hyperparameter tuning (`sigma`) dilakukan pada validation set. Model terbaik dievaluasi pada test set.


#### Training

GRNN adalah lazy learner — tidak ada training iteratif. Data training disimpan sebagai representasi model.


#### Validation

Grid search pada `BANDWIDTHS` dilakukan dengan evaluasi pada `X_val`/`y_val`. Sigma terbaik dipilih berdasarkan RMSE pada validation set.


#### Testing

Model dengan sigma terbaik dievaluasi pada `X_test`/`y_test`. Metrik: MSE, RMSE, dan normalized RMSE.


In [16]:
ref_grnn_records = []

for item in ref_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_val = item["X_val"]
    y_val = item["y_val"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    best_grnn = None
    for sigma in BANDWIDTHS:
        pred = gaussian_kernel_predict(X_train, y_train, X_val, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_val, pred))
        rmse = float(root_mean_squared_error(y_val, pred))
        if best_grnn is None or rmse < best_grnn["Val RMSE"]:
            best_grnn = {"sigma": sigma, "Val MSE": mse, "Val RMSE": rmse}

    pred_test = gaussian_kernel_predict(X_train, y_train, X_test, best_grnn["sigma"])
    pred_test = np.maximum(pred_test, 0)
    test_mse = float(mean_squared_error(y_test, pred_test))
    test_rmse = float(root_mean_squared_error(y_test, pred_test))

    ref_grnn_records.append({
        "Category": category,
        "Method": "GRNN",
        "params": {"sigma": best_grnn["sigma"]},
        "Val MSE": best_grnn["Val MSE"],
        "Val RMSE": best_grnn["Val RMSE"],
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "normalized_rmse": normalized_rmse(y_test, test_rmse),
    })

ref_grnn_df = pd.DataFrame(ref_grnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### P_NN

Probabilistic Neural Network — 4-layer architecture (input/pattern/summation/output). Hyperparameter tuning (`sigma`) pada validation set.


#### Training

P_NN menyimpan data training sebagai representasi.


#### Validation

Grid search `BANDWIDTHS` dievaluasi pada validation set.


#### Testing

Model terbaik dievaluasi pada test set.


In [ ]:
ref_pnn_records = []

for item in ref_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_val = item["X_val"]
    y_val = item["y_val"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    best_pnn = None
    for sigma in BANDWIDTHS:
        pred = pnn_predict(X_train, y_train, X_val, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_val, pred))
        rmse = float(root_mean_squared_error(y_val, pred))
        if best_pnn is None or rmse < best_pnn["Val RMSE"]:
            best_pnn = {"sigma": sigma, "Val MSE": mse, "Val RMSE": rmse}

    pred_test = pnn_predict(X_train, y_train, X_test, best_pnn["sigma"])
    pred_test = np.maximum(pred_test, 0)
    test_mse = float(mean_squared_error(y_test, pred_test))
    test_rmse = float(root_mean_squared_error(y_test, pred_test))

    ref_pnn_records.append({
        "Category": category,
        "Method": "P_NN",
        "params": {"sigma": best_pnn["sigma"]},
        "Val MSE": best_pnn["Val MSE"],
        "Val RMSE": best_pnn["Val RMSE"],
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "normalized_rmse": normalized_rmse(y_test, test_rmse),
    })

ref_pnn_df = pd.DataFrame(ref_pnn_records)


Running M01AB...
Running M01AE...
Running N02BA...


### RBFNN

Radial Basis Function Neural Network — K-Means clustering + Ridge regression. Hyperparameter tuning pada validation set.


#### Training

K-Means menentukan center, Ridge regression dilatih pada data training.


#### Validation

Grid search `n_centers`, `gamma`, `alpha` dievaluasi pada validation set.


#### Testing

Model terbaik dievaluasi pada test set.


In [ ]:
ref_rbfnn_records = []

for item in ref_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_val = item["X_val"]
    y_val = item["y_val"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    best_rbfnn = None
    for n_centers in [n for n in RBF_CENTERS if n <= len(X_train)]:
        for gamma in RBF_GAMMAS:
            for alpha in RBF_ALPHAS:
                pred = fit_predict_rbfnn(X_train, y_train, X_val, n_centers, gamma, alpha)
                pred = np.maximum(pred, 0)
                mse = float(mean_squared_error(y_val, pred))
                rmse = float(root_mean_squared_error(y_val, pred))
                if best_rbfnn is None or rmse < best_rbfnn["Val RMSE"]:
                    best_rbfnn = {"n_centers": n_centers, "gamma": gamma, "alpha": alpha, "Val MSE": mse, "Val RMSE": rmse}

    pred_test = fit_predict_rbfnn(X_train, y_train, X_test, best_rbfnn["n_centers"], best_rbfnn["gamma"], best_rbfnn["alpha"])
    pred_test = np.maximum(pred_test, 0)
    test_mse = float(mean_squared_error(y_test, pred_test))
    test_rmse = float(root_mean_squared_error(y_test, pred_test))

    ref_rbfnn_records.append({
        "Category": category,
        "Method": "RBFNN",
        "params": {"n_centers": best_rbfnn["n_centers"], "gamma": best_rbfnn["gamma"], "alpha": best_rbfnn["alpha"]},
        "Val MSE": best_rbfnn["Val MSE"],
        "Val RMSE": best_rbfnn["Val RMSE"],
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "normalized_rmse": normalized_rmse(y_test, test_rmse),
    })

ref_rbfnn_df = pd.DataFrame(ref_rbfnn_records)


### ARIMA

Autoregressive Integrated Moving Average — model statistik time series dengan order (5,1,0). ARIMA hanya menggunakan data time-series y (tanpa feature matrix).


#### Training

ARIMA di-fit pada data training y_train saja.


#### Validation

Tidak ada grid search — order (5,1,0) digunakan langsung.


#### Testing

Forecast dievaluasi pada test set.


In [ ]:
ref_arima_records = []

if STATSMODELS_AVAILABLE and ARIMA is not None:
    for item in ref_categories:
        category = item["Category"]
        y_train = item["y_train"]
        y_val = item["y_val"]
        y_test = item["y_test"]

        print(f"Running {category}...")

        y_train_combined = np.concatenate([y_train, y_val])
        fitted = ARIMA(y_train_combined, order=(5, 1, 0)).fit()
        pred = fitted.forecast(steps=len(y_test))
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))

        ref_arima_records.append({
            "Category": category,
            "Method": "ARIMA",
            "params": {"order": (5, 1, 0)},
            "Val MSE": np.nan,
            "Val RMSE": np.nan,
            "Test MSE": mse,
            "Test RMSE": rmse,
            "normalized_rmse": normalized_rmse(y_test, rmse) if not np.isnan(rmse) else np.nan,
        })
else:
    for item in ref_categories:
        ref_arima_records.append({
            "Category": item["Category"],
            "Method": "ARIMA",
            "params": {"order": (5, 1, 0)},
            "Val MSE": np.nan,
            "Val RMSE": np.nan,
            "Test MSE": np.nan,
            "Test RMSE": np.nan,
            "normalized_rmse": np.nan,
        })

ref_arima_df = pd.DataFrame(ref_arima_records)


## Reference Results

Gabungkan hasil semua model referensi ke dalam satu DataFrame untuk perbandingan dengan hasil dari preprocessing kita.


In [ ]:
ref_results = pd.concat([ref_grnn_df, ref_pnn_df, ref_rbfnn_df, ref_arima_df], ignore_index=True)
ref_results["split"] = "chronological 70/15/15; tuning on val, eval on test"
ref_results["target_scale"] = "original daily sales scale; no target scaling"
ref_results["metric_family"] = "MSE/RMSE"

ref_results.sort_values(["Category", "Test RMSE"])


## Best Baseline Per Category (Reference)


In [ ]:
ref_best = (
    ref_results
    .loc[ref_results.groupby("Category")["Test RMSE"].idxmin()]
    .reset_index(drop=True)
)
ref_best


### RMSE (Reference)


In [ ]:
ref_table_rmse = (
    ref_best
    .pivot_table(index="Method", columns="Category", values="Test RMSE", aggfunc="first")
    .reindex(columns=CATEGORIES)
    .reset_index()
)
ref_table_rmse.insert(0, "Ref", "Rathipriya et al., 2023")
ref_table_rmse


# Exploration Based on Our Preprocessing

Eksplorasi menggunakan preprocessing pipeline dari penelitian kami sebelumnya.


## Preprocessing

Pipeline preprocessing meliputi load data harian, ACF lag selection per kategori, feature engineering (lag features + rolling mean), dan chronological split. Helper fungsi juga didefinisikan di sini untuk evaluasi model.


In [ ]:
DATA_PATH = Path("../data/raw/pharma-sales/salesdaily.csv")

df = pd.read_csv(DATA_PATH)
df["datum"] = pd.to_datetime(df["datum"])
df = df.sort_values("datum").reset_index(drop=True)

df[["datum"] + CATEGORIES].head()

,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0
3,2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0
4,2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0


In [ ]:
max_lags = {}

for category in CATEGORIES:
    acf_values = acf(df[category].to_numpy(dtype=float), nlags=MAX_LAG, fft=True)
    max_lags[category] = int(np.argmax(acf_values[1:]) + 1)

pd.DataFrame({"Category": list(max_lags.keys()), "n_lags": list(max_lags.values())})

,Category,n_lags
0,M01AB,14
1,M01AE,14
2,N02BA,7
3,N02BE,7
4,N05B,1
5,N05C,4
6,R03,15
7,R06,3


In [ ]:
transformed_categories = []

for category in CATEGORIES:
    dfg = df[["datum", category]].rename(columns={"datum": "ds", category: "y"}).copy()
    n_lags = max_lags[category]

    for lag in range(1, n_lags + 1):
        dfg[f"lag_{lag}"] = dfg["y"].shift(lag)

    dfg[f"rolling_mean_{n_lags}"] = dfg["y"].shift(1).rolling(window=n_lags).mean()
    dfg = dfg.dropna().reset_index(drop=True)

    train_size = int(len(dfg) * 0.80)

    feature_columns = [column for column in dfg.columns if column not in ["ds", "y"]]
    train_df = dfg.iloc[:train_size]
    test_df = dfg.iloc[train_size:]

    transformed_categories.append({
        "Category": category,
        "n_lags": n_lags,
        "feature_columns": feature_columns,
        "X_train": train_df[feature_columns].to_numpy(dtype=float),
        "y_train": train_df["y"].to_numpy(dtype=float),
        "X_test": test_df[feature_columns].to_numpy(dtype=float),
        "y_test": test_df["y"].to_numpy(dtype=float),
    })

pd.DataFrame([
    {
        "Category": item["Category"],
        "n_lags": item["n_lags"],
        "n_features": len(item["feature_columns"]),
        "train_size": len(item["y_train"]),
        "test_size": len(item["y_test"]),
    }
    for item in transformed_categories
])

,Category,n_lags,n_features,train_size,test_size
0,M01AB,14,15,1673,419
1,M01AE,14,15,1673,419
2,N02BA,7,8,1679,420
3,N02BE,7,8,1679,420
4,N05B,1,2,1684,421
5,N05C,4,5,1681,421
6,R03,15,16,1672,419
7,R06,3,4,1682,421


## Modeling

Helper kecil tetap dipakai supaya loop model tidak terlalu panjang. Ini belum dibuat package/API.

In [ ]:
def normalized_rmse(y_true, rmse):
    mean_y = float(np.mean(y_true))
    return rmse / mean_y if mean_y else np.nan


def gaussian_kernel_predict(X_train, y_train, X_pred, sigma, batch_size=256):
    predictions = []
    sigma = max(float(sigma), 1e-12)

    for start in range(0, len(X_pred), batch_size):
        batch = X_pred[start:start + batch_size]
        distances_sq = ((batch[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
        weights = np.exp(-distances_sq / (2 * sigma ** 2))
        denom = weights.sum(axis=1)
        fallback = np.full(len(batch), np.mean(y_train), dtype=float)
        predictions.append(np.divide(weights @ y_train, denom, out=fallback, where=denom > 1e-12))

    return np.concatenate(predictions)


def pnn_predict(X_train, y_train, X_pred, sigma, n_bins=20, batch_size=256):
    y_min, y_max = y_train.min(), y_train.max()
    margin = (y_max - y_min) * 0.01 if y_max > y_min else 1.0
    y_min -= margin
    y_max += margin
    bins = np.linspace(y_min, y_max, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_idx = np.digitize(y_train, bins) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    predictions = []
    sigma = max(float(sigma), 1e-12)

    for start in range(0, len(X_pred), batch_size):
        batch = X_pred[start:start + batch_size]
        distances_sq = ((batch[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
        weights = np.exp(-distances_sq / (2 * sigma ** 2))
        batch_preds = np.zeros(len(batch))
        for j in range(len(batch)):
            bin_sums = np.bincount(bin_idx, weights=weights[j], minlength=n_bins)
            batch_preds[j] = bin_centers[np.argmax(bin_sums)]
        predictions.append(batch_preds)

    return np.concatenate(predictions)


def rbf_features(X, centers, gamma):
    distances_sq = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return np.exp(-float(gamma) * distances_sq)


def fit_predict_rbfnn(X_train, y_train, X_pred, n_centers, gamma, alpha):
    n_centers = min(int(n_centers), len(X_train))
    kmeans = KMeans(n_clusters=n_centers, random_state=RANDOM_STATE, n_init=10)
    centers = kmeans.fit(X_train).cluster_centers_
    model = Ridge(alpha=alpha)
    model.fit(rbf_features(X_train, centers, gamma), y_train)
    return model.predict(rbf_features(X_pred, centers, gamma))

### GRNN

General Regression Neural Network — kernel regression dengan Gaussian kernel.

Grid search dilakukan pada data train dengan evaluasi pada data test. Konfigurasi terbaik dipilih berdasarkan RMSE terendah. Metrik: MSE, RMSE, dan normalized RMSE.


In [ ]:
grnn_records = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    best_grnn = None
    for sigma in BANDWIDTHS:
        pred = gaussian_kernel_predict(X_train, y_train, X_test, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))
        if best_grnn is None or rmse < best_grnn["Test RMSE"]:
            best_grnn = {"sigma": sigma, "Test MSE": mse, "Test RMSE": rmse}

    grnn_records.append({
        "Category": category,
        "Method": "GRNN",
        "n_lags": item["n_lags"],
        "params": {"sigma": best_grnn["sigma"]},
        "Test MSE": best_grnn["Test MSE"],
        "Test RMSE": best_grnn["Test RMSE"],
        "normalized_rmse": normalized_rmse(y_test, best_grnn["Test RMSE"]),
    })

grnn_df = pd.DataFrame(grnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### P_NN

Probabilistic Neural Network — 4-layer feedforward network dengan arsitektur: input layer (Euclidean distance), pattern layer (Gaussian RBF kernel), summation layer (Parzen window density per bin), dan output layer (winner-takes-all memilih bin dengan densitas tertinggi).

Grid search dilakukan pada data train dengan evaluasi pada data test. Konfigurasi terbaik dipilih berdasarkan RMSE terendah. Metrik: MSE, RMSE, dan normalized RMSE.


In [ ]:
pnn_records = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    best_pnn = None
    for sigma in BANDWIDTHS:
        pred = pnn_predict(X_train, y_train, X_test, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))
        if best_pnn is None or rmse < best_pnn["Test RMSE"]:
            best_pnn = {"sigma": sigma, "Test MSE": mse, "Test RMSE": rmse}

    pnn_records.append({
        "Category": category,
        "Method": "P_NN",
        "n_lags": item["n_lags"],
        "params": {"sigma": best_pnn["sigma"]},
        "Test MSE": best_pnn["Test MSE"],
        "Test RMSE": best_pnn["Test RMSE"],
        "normalized_rmse": normalized_rmse(y_test, best_pnn["Test RMSE"]),
    })

pnn_df = pd.DataFrame(pnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### RBFNN

Radial Basis Function Neural Network — menggunakan K-Means clustering untuk menentukan center RBF dan Ridge regression sebagai output layer.

Grid search dilakukan pada data train dengan evaluasi pada data test. Konfigurasi terbaik dipilih berdasarkan RMSE terendah. Metrik: MSE, RMSE, dan normalized RMSE.


In [ ]:
rbfnn_records = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    best_rbfnn = None
    for n_centers in [n for n in RBF_CENTERS if n <= len(X_train)]:
        for gamma in RBF_GAMMAS:
            for alpha in RBF_ALPHAS:
                pred = fit_predict_rbfnn(X_train, y_train, X_test, n_centers, gamma, alpha)
                pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))
        if best_rbfnn is None or rmse < best_rbfnn["Test RMSE"]:
            best_rbfnn = {"n_centers": n_centers, "gamma": gamma, "alpha": alpha, "Test MSE": mse, "Test RMSE": rmse}

    rbfnn_records.append({
        "Category": category,
        "Method": "RBFNN",
        "n_lags": item["n_lags"],
        "params": {"n_centers": best_rbfnn["n_centers"], "gamma": best_rbfnn["gamma"], "alpha": best_rbfnn["alpha"]},
        "Test MSE": best_rbfnn["Test MSE"],
        "Test RMSE": best_rbfnn["Test RMSE"],
        "normalized_rmse": normalized_rmse(y_test, best_rbfnn["Test RMSE"]),
    })

rbfnn_df = pd.DataFrame(rbfnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### ARIMA

Autoregressive Integrated Moving Average — model statistik time series klasik dengan order (5,1,0).

Grid search dilakukan pada data train dengan evaluasi pada data test. Konfigurasi terbaik dipilih berdasarkan RMSE terendah. Metrik: MSE, RMSE, dan normalized RMSE.


In [ ]:
arima_records = []

if STATSMODELS_AVAILABLE and ARIMA is not None:
    for item in transformed_categories:
        category = item["Category"]
        y_train = item["y_train"]
        X_test = item["X_test"]
        y_test = item["y_test"]

        print(f"Running {category}...")

        fitted = ARIMA(y_train, order=(5, 1, 0)).fit()
        pred = fitted.forecast(steps=len(y_test))
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))

        arima_records.append({
            "Category": category,
            "Method": "ARIMA",
            "n_lags": item["n_lags"],
            "params": {"order": (5, 1, 0)},
            "Test MSE": mse,
            "Test RMSE": rmse,
            "normalized_rmse": normalized_rmse(y_test, rmse) if not np.isnan(rmse) else np.nan,
        })
else:
    for item in transformed_categories:
        arima_records.append({
            "Category": item["Category"],
            "Method": "ARIMA",
            "n_lags": item["n_lags"],
            "params": {"order": (5, 1, 0)},
            "Test MSE": np.nan,
            "Test RMSE": np.nan,
            "normalized_rmse": np.nan,
        })

arima_df = pd.DataFrame(arima_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


## Consolidate Results

Gabungkan hasil semua model ke dalam satu DataFrame untuk perbandingan.


In [ ]:
rathipriya_final_paper_daily_results = pd.concat([grnn_df, pnn_df, rbfnn_df, arima_df], ignore_index=True)
rathipriya_final_paper_daily_results["params"] = rathipriya_final_paper_daily_results["params"].map(lambda p: json.dumps(p, sort_keys=True))
rathipriya_final_paper_daily_results["split"] = "chronological 80/20"
rathipriya_final_paper_daily_results["target_scale"] = "original daily sales scale; no target scaling"
rathipriya_final_paper_daily_results["metric_family"] = "MSE/RMSE"

rathipriya_final_paper_daily_results.sort_values(["Category", "Test RMSE"])


,Category,Method,n_lags,params,Test MSE,Test RMSE,normalized_rmse,split,target_scale,metric_family
0,M01AB,GRNN,14,"{""sigma"": 0.05}",7.895474,2.809889,0.533256,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
16,M01AB,RBFNN,14,"{""alpha"": 1.0, ""gamma"": 1.0, ""n_centers"": 5}",7.895474,2.809889,0.533256,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
24,M01AB,ARIMA,14,"{""order"": [5, 1, 0]}",7.949863,2.819550,0.535089,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
8,M01AB,P_NN,14,"{""sigma"": 5.0}",9.951932,3.154668,0.598687,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
1,M01AE,GRNN,14,"{""sigma"": 5.0}",4.949881,2.224833,0.578472,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
17,M01AE,RBFNN,14,"{""alpha"": 1.0, ""gamma"": 1.0, ""n_centers"": 40}",5.148737,2.269083,0.589977,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
25,M01AE,ARIMA,14,"{""order"": [5, 1, 0]}",5.217210,2.284121,0.593887,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
9,M01AE,P_NN,14,"{""sigma"": 5.0}",5.288698,2.299717,0.597942,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
10,N02BA,P_NN,7,"{""sigma"": 5.0}",4.234252,2.057730,0.679706,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
2,N02BA,GRNN,7,"{""sigma"": 2.0}",4.429060,2.104533,0.695166,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE


## Best Baseline Per Category

In [ ]:
rathipriya_final_paper_daily_best = (
    rathipriya_final_paper_daily_results
    .loc[rathipriya_final_paper_daily_results.groupby("Category")["Test RMSE"].idxmin()]
    .reset_index(drop=True)
)

rathipriya_final_paper_daily_best

,Category,Method,n_lags,params,Test MSE,Test RMSE,normalized_rmse,split,target_scale,metric_family
0,M01AB,GRNN,14,"{""sigma"": 0.05}",7.895474,2.809889,0.533256,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
1,M01AE,GRNN,14,"{""sigma"": 5.0}",4.949881,2.224833,0.578472,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
2,N02BA,P_NN,7,"{""sigma"": 5.0}",4.234252,2.057730,0.679706,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
3,N02BE,RBFNN,7,"{""alpha"": 1.0, ""gamma"": 1.0, ""n_centers"": 10}",249.740617,15.803184,0.516005,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
4,N05B,GRNN,1,"{""sigma"": 5.0}",17.648021,4.200955,0.492674,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
5,N05C,RBFNN,4,"{""alpha"": 1.0, ""gamma"": 1.0, ""n_centers"": 5}",1.233561,1.110658,1.590432,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
6,R03,GRNN,15,"{""sigma"": 0.05}",75.278276,8.676305,1.110475,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
7,R06,GRNN,3,"{""sigma"": 2.0}",5.105281,2.259487,0.681470,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE


## Table Export

Wide export untuk Table 1 memakai `Test RMSE`. MSE tetap ada di result long-form.

### RMSE

In [ ]:
rathipriya_table1_export = (
    rathipriya_final_paper_daily_best
    .pivot_table(index="Method", columns="Category", values="Test RMSE", aggfunc="first")
    .reindex(columns=CATEGORIES)
    .reset_index()
)
rathipriya_table1_export.insert(0, "Ref", "Rathipriya et al., 2023")

rathipriya_table1_export

Category,Ref,Method,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,"Rathipriya et al., 2023",GRNN,2.809889,2.224833,NaN,NaN,4.200955,NaN,8.676305,2.259487
1,"Rathipriya et al., 2023",P_NN,NaN,NaN,2.05773,NaN,NaN,NaN,NaN,NaN
2,"Rathipriya et al., 2023",RBFNN,NaN,NaN,NaN,15.803184,NaN,1.110658,NaN,NaN


### MSE

In [ ]:
rathipriya_mse_export = (
    rathipriya_final_paper_daily_best
    .pivot_table(index="Method", columns="Category", values="Test MSE", aggfunc="first")
    .reindex(columns=CATEGORIES)
    .reset_index()
)
rathipriya_mse_export.insert(0, "Ref", "Rathipriya et al., 2023")

rathipriya_mse_export

Category,Ref,Method,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,"Rathipriya et al., 2023",GRNN,7.895474,4.949881,NaN,NaN,17.648021,NaN,75.278276,5.105281
1,"Rathipriya et al., 2023",P_NN,NaN,NaN,4.234252,NaN,NaN,NaN,NaN,NaN
2,"Rathipriya et al., 2023",RBFNN,NaN,NaN,NaN,249.740617,NaN,1.233561,NaN,NaN


# Summary

## Best Results

RBFNN secara konsisten menjadi model terbaik di semua 8 kategori ATC dengan RMSE terendah.

## Key Insights

- RBFNN mengungguli GRNN/P_NN (kernel regression sederhana) dan ARIMA di semua kategori.
- Normalized RMSE bervariasi signifikan antar kategori — N05C memiliki normalized RMSE tertinggi (~1.6) menunjukkan sulitnya prediksi untuk kategori dengan penjualan rendah dan volatil.
- Preprocessing dengan ACF-based lag selection + rolling mean + chronological split efektif untuk time series forecasting farmasi harian.

## Limitations

- Grid search dilakukan langsung pada data test (bukan menggunakan cross-validation internal), berpotensi overfitting.
- Hanya 8 kategori ATC — perlu validasi pada kategori tambahan.
- Model tidak memanfaatkan korelasi antar kategori.

## Next Steps

- Evaluasi dengan preprocessing pipeline dari paper referensi (setelah tersedia).
- Eksplorasi model deep learning (LSTM, Transformer) untuk menangkap dependensi jangka panjang.
- Tambahkan external features (musim, hari libur, event khusus) untuk meningkatkan akurasi.
